In [13]:
import os
import re

import numpy as np
import torch
import pandas as pd
from val_test import val_test
from print_results import *
import pickle
from tqdm.notebook import tqdm
from functionsgpu_old import *
from plotting_betas import *
import torch
import torch.nn as nn
import torch.nn.functional as F
from video_saving import *

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)
dtype = torch.float32

if device.type == "cuda":
    idx = device.index if device.index is not None else torch.cuda.current_device()
    print(torch.cuda.get_device_name(idx))

SEED = 42
def deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

deterministic(SEED)
# Enable (as much as possible) deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tslen = 200

cuda:1
NVIDIA RTX A5000


## Load POMA and Participant IDs

In [7]:
# Load POMA scores and participant IDs
# Participant ID for each row of dataset (same order as files from csv_r)
participant_ids = np.loadtxt('pids.txt')
y_poma = np.loadtxt('y_poma.txt')

print("y_poma shape:", y_poma.shape)
print("First 10 participant_ids:", participant_ids[:10])

y_poma shape: (155,)
First 10 participant_ids: [ 1.  2.  3.  4.  5.  6.  7.  8. 10. 11.]


## Data Loading

In [8]:
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all

betas_aligned_all, mu_all_t, tangent_vec_all = loading('aligned_data',tslen)
mu_all_t_tensor = torch.from_numpy(mu_all_t).to(device=device, dtype=torch.float32)
betas_aligned = np.array(betas_aligned_all)
betas_aligned = betas_aligned.transpose(1, 2, 3, 0)
print(betas_aligned.shape, tangent_vec_all.shape, mu_all_t.shape)

(32, 3, 200, 155) (32, 3, 200, 155) (32, 3, 200)


In [9]:
K = 32
M = 3
T = tslen
nsamples = 155

tangent_flat = tangent_vec_all.reshape((K*M*T, nsamples))
print(tangent_flat.shape)

(19200, 155)


# Nonlinear Tangent VAE

In [10]:
class NonlinearVAE(nn.Module):
    """VAE with the same nonlinear/tied architecture as NonlinearAE.

    Encoder: x -> W1 -> tanh -> dropout -> {W2_mu, W2_logvar} -> {mu, logvar}
    Decoder (tied): z -> W2_mu^T -> tanh -> W1^T -> x_hat
    """
    def __init__(self, D, R, H=32, dropout=0.1):
        super().__init__()
        self.H = H
        # Encoder layers
        self.W1 = nn.Linear(D, H, bias=False)        # input -> hidden
        self.W2_mu = nn.Linear(H, R, bias=False)     # hidden -> latent mean
        self.W2_logvar = nn.Linear(H, R)             # hidden -> latent logvar
        self.dropout = nn.Dropout(p=dropout)

    def encode(self, x):
        h = torch.tanh(self.W1(x))
        h = self.dropout(h)
        mu = self.W2_mu(h)
        logvar = self.W2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        # Tied decoder using W2_mu and W1, mirroring NonlinearAE
        h_recon = torch.tanh(F.linear(z, self.W2_mu.weight.T))
        x_hat = F.linear(h_recon, self.W1.weight.T)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def vae_loss(x, x_hat, mu, logvar, beta=1e-4):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


## Training Function for Each Fold Training (VAE)

In [8]:
def train_vae_fold(X_tan_train, D, R, num_epochs=1000, lr=1e-3, verbose=False, seed=42):
    """Train a fresh NonlinearVAE on a training subset only (tangent space MSE + KL loss).

    X_tan_train : (N_train, D) tangent vectors
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    model_fold.train()
    for epoch in range(num_epochs):
        opt_fold.zero_grad(set_to_none=True)
        x_hat, mu, logvar, z = model_fold(X_tan_train)
        loss_train, recon_train, kl_train = vae_loss(X_tan_train, x_hat, mu, logvar, beta=1e-5)
        loss_train.backward()
        opt_fold.step()
        if verbose and (epoch % 300 == 0 or epoch == num_epochs - 1):
            print(f"[fold VAE] epoch {epoch} | loss {loss_train:.6f} | recon {recon_train:.6f} | kl {kl_train:.6f}")
    model_fold.eval()
    return model_fold

## VAE Cross Validation

In [9]:
# VAE latent regression: same CV as RVAE (5 val + 5 test per fold, two rounds)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb

dtype = torch.float32
n = len(y_poma)
D = tangent_flat.shape[0]
n_folds = 30
R_vae = 38

models = {
    'KNN': KNeighborsRegressor(),
    'SVM': SVR(kernel='rbf'),
    'RF': RandomForestRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42),
    'MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}
all_results_validation_vae = {name: {'targets': [], 'preds': []} for name in models.keys()}
all_results_test_vae = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)


for k in tqdm(range(n_folds), total=n_folds, desc='VAE folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
        continue

    fold_seed = SEED + k
    X_tan_train = torch.from_numpy(tangent_flat[:, train_idx].T.astype(np.float32)).to(device=device, dtype=dtype)
    model_fold = train_vae_fold(X_tan_train, D, R_vae, num_epochs=2000, lr=1e-3, verbose=True, seed=fold_seed)

    with torch.no_grad():
        mu_train_fold, _ = model_fold.encode(X_tan_train)
        mu_val_fold, _ = model_fold.encode(torch.from_numpy(tangent_flat[:, validation_idx].T.astype(np.float32)).to(device=device, dtype=dtype))
        mu_test_fold, _ = model_fold.encode(torch.from_numpy(tangent_flat[:, test_idx].T.astype(np.float32)).to(device=device, dtype=dtype))

    Z_train_fold = mu_train_fold.cpu().numpy()
    Z_val_fold = mu_val_fold.cpu().numpy()
    Z_test_fold = mu_test_fold.cpu().numpy()
    y_train_fold = y_poma[train_idx]
    y_val_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    for name, model_reg in models.items():
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)
        validation_preds = m.predict(Z_val_fold)
        test_preds = m.predict(Z_test_fold)
        all_results_validation_vae[name]['targets'].extend(y_val_fold.tolist())
        all_results_validation_vae[name]['preds'].extend(validation_preds.tolist())
        all_results_test_vae[name]['targets'].extend(y_test_fold.tolist())
        all_results_test_vae[name]['preds'].extend(test_preds.tolist())
        all_results_test_vae[name]['subjects'].extend(participant_ids[test_idx].tolist())
        mae_val = mean_absolute_error(y_val_fold, validation_preds)
        rmse_val = np.sqrt(mean_squared_error(y_val_fold, validation_preds))
        r2_val = r2_score(y_val_fold, validation_preds)
        print(f"Fold {k + 1:02d} | {name} | Validation: MAE={mae_val:.3f}, RMSE={rmse_val:.3f}, R2={r2_val:.3f}")

results_test_vae_df = print_results(all_results_validation_vae, all_results_test_vae, models)
results_test_vae_df

VAE folds:   0%|          | 0/30 [00:00<?, ?it/s]

[fold VAE] epoch 0 | loss 0.000268 | recon 0.000268 | kl 0.001727
[fold VAE] epoch 300 | loss 0.000123 | recon 0.000123 | kl 0.021788
[fold VAE] epoch 600 | loss 0.000109 | recon 0.000109 | kl 0.046379
[fold VAE] epoch 900 | loss 0.000098 | recon 0.000097 | kl 0.087506
[fold VAE] epoch 1200 | loss 0.000095 | recon 0.000094 | kl 0.104181
[fold VAE] epoch 1500 | loss 0.000090 | recon 0.000089 | kl 0.130383
[fold VAE] epoch 1800 | loss 0.000087 | recon 0.000086 | kl 0.143709
[fold VAE] epoch 1999 | loss 0.000086 | recon 0.000085 | kl 0.148944
Fold 01 | KNN | Validation: MAE=2.600, RMSE=3.680, R2=0.436
Fold 01 | SVM | Validation: MAE=2.503, RMSE=3.036, R2=0.616
Fold 01 | RF | Validation: MAE=2.352, RMSE=2.760, R2=0.683
Fold 01 | XGBoost | Validation: MAE=3.098, RMSE=3.519, R2=0.484
Fold 01 | MLP | Validation: MAE=2.397, RMSE=3.326, R2=0.539
[fold VAE] epoch 0 | loss 0.000266 | recon 0.000266 | kl 0.002266
[fold VAE] epoch 300 | loss 0.000118 | recon 0.000118 | kl 0.028561
[fold VAE] epoch 

,MAE,RMSE,R2,Pearson r,Pearson p
KNN,1.330323,2.893775,0.725796,0.853020,4.622594e-45
SVM,1.515977,3.208289,0.662952,0.847629,5.856798e-44
RF,1.469484,3.230743,0.658218,0.817104,1.978933e-38
XGBoost,1.608256,3.625975,0.569479,0.785859,9.477359e-34
MLP,1.666291,3.194038,0.665940,0.818458,1.184676e-38


In [10]:
from ci import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_test_vae[name]['targets'],
    all_results_test_vae[name]['preds'],
    all_results_test_vae[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,1.330323,2.893775,0.725796,0.85302
ci,"[1.018, 1.635]","[2.339, 3.308]","[0.647, 0.803]","[0.809, 0.9]"


# KT-RSV

In [14]:
class KendallRVAE(nn.Module):
    """rVAE: encode tangent vectors, decode to tangent, but (during
    training) compare reconstructions on the manifold via an exp map.

    - Inputs: tangent vectors (N, D) as in the VAE section.
    - Decoder: produces tangent vectors.
    - Training loss: uses expmap(mu, v_hat) vs. original manifold trajectory.
    """
    def __init__(self, base_vae, mu_shape, expmap):
        super().__init__()
        self.vae = base_vae
        # mean shape, used for exponential map when mapping back to manifold
        self.register_buffer("mu_shape", mu_shape)
        self.expmap = expmap

    def forward(self, x):
        """Forward on tangent vectors.

        x : (N, D) tangent vectors
        Returns
        -------
        x_man_hat : (N, D) manifold trajectory flattened (via expmap)
        mu_z, logvar, z, v_hat : usual VAE outputs (in tangent space)
        """
        v_hat, mu_z, logvar, z = self.vae(x)   # tangent reconstruction

        # Map reconstructed tangent field back to the manifold
        B = v_hat.shape[0]
        v_hat_reshaped = v_hat.view(B, K, M, T)
        mu = self.mu_shape.view(K, M, T)
        x_recon_man = self.expmap(mu, v_hat_reshaped)   # (B, K, M, T)
        x_recon_man = x_recon_man.view(B, -1)

        return x_recon_man, mu_z, logvar, z, v_hat

## Getting Orginal Trajectories on Manifold

In [15]:
betas = np.array(betas_aligned_all)
print(betas.shape)   # (155, 32, 3, 200)

N, K, M, T = betas.shape   # correct order

# Shape mean as before (kept for potential manifold utilities)
mu_shape = torch.from_numpy(
    mu_all_t.reshape(-1).astype(np.float32)
).to(device)

print("mu_shape:", mu_shape.shape)      # should be (19200,)

(155, 32, 3, 200)
mu_shape: torch.Size([19200])


## Training on Full Dataset (test run)

In [ ]:
# Tangent input (as in the VAE section)
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
mu_shape = mu_shape.to(device=device, dtype=dtype)

# Corresponding manifold trajectories (flattened)
betas = np.array(betas_aligned_all)              # (N, K, M, T)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

D = X_tan.shape[1]
R = 38

base_vae = NonlinearVAE(D, R).to(device=device, dtype=dtype)
model = KendallRVAE(base_vae, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3000):
    opt.zero_grad(set_to_none=True)

    # forward in tangent space, loss in manifold space
    x_hat_man, mu, logvar, z, v_hat = model(X_tan)

    # Geodesic reconstruction loss on the manifold
    dist = squared_geodesic_distance(X_man, x_hat_man, K, M, T)
    recon = torch.mean(dist)

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # loss = recon + 1e-6 * kl
    loss = recon + 1e-6 * kl

    loss.backward()
    opt.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | loss {loss:.6f} | recon {recon:.6f} | kl {kl:.6f}")

Epoch 0 | loss 0.026149 | recon 0.026149 | kl 0.001728


KeyboardInterrupt: 

In [ ]:
%matplotlib qt5
plot_one_traj(x_hat_man[1,:].reshape(32,3,tslen))

libGL error: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
libGL error: failed to load driver: swrast


## Training Function for Each Fold Training (KT-RSV)

In [18]:
def train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T, R=10,
                             num_epochs=2000, lr=1e-3, verbose=False, seed=42):
    """Train a fresh KendallRVAE on a training subset only.

    X_tan_train : (N_train, D) tangent vectors
    X_man_train : (N_train, K*M*T) manifold trajectories (flattened)
    seed : random seed for model initialization
    """
    # Set seeds before model initialization for reproducibility
    deterministic(seed)
    
    D = X_tan_train.shape[1]

    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = KendallRVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    model_fold.train()
    for epoch in range(num_epochs):
        opt_fold.zero_grad(set_to_none=True)

        x_hat_man_train, mu_train, logvar_train, z_train, v_hat_train = model_fold(X_tan_train)

        dist_train = squared_geodesic_distance(X_man_train, x_hat_man_train, K, M, T)
        recon_train = torch.mean(dist_train)

        kl_train = -0.5 * torch.mean(1 + logvar_train - mu_train.pow(2) - logvar_train.exp())
        loss_train = recon_train + 1e-6 * kl_train
        # loss_train = recon_train + 1e-5 * kl_train + lambda_cov * cov_penalty(mu_train)

        loss_train.backward()
        opt_fold.step()

        if verbose and (epoch % 300 == 0 or epoch == num_epochs - 1):
            print(f"[fold RVAE] epoch {epoch} | loss {loss_train:.6f} | recon {recon_train:.6f} | kl {kl_train:.6f}")

    model_fold.eval()
    return model_fold

## KT-RSV Cross Validation

In [19]:
# Nested CV: re-train KendallRVAE inside each participant fold

# Set random seeds for reproducibility
import random
import numpy as np
import torch

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from tqdm.notebook import tqdm


# Regression models with random_state set for reproducibility
models = {'KNN': KNeighborsRegressor()}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_poma)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 30
R = 38
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    deterministic(fold_seed)

    # Slice tangent and manifold data for this fold
    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]

    # Train fold-specific KendallRVAE on train participants only
    model_fold = train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T,
                                         R=R, num_epochs=2000, lr=1e-3, verbose=True, seed=fold_seed)

    # Extract latent means (mu) for train, validation, and test using the fold-specific encoder
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_validation_fold, _ = model_fold.vae.encode(X_tan[validation_idx])
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_validation_fold = mu_validation_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    y_train_fold = y_poma[train_idx]
    y_validation_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold validation metrics (printed)
        mae_validation_fold = mean_absolute_error(y_validation_fold, validation_preds)
        rmse_validation_fold = np.sqrt(mean_squared_error(y_validation_fold, validation_preds))
        r2_validation_fold = r2_score(y_validation_fold, validation_preds)

        print(
            f"Fold {k + 1:02d} | {name} | Validation: "
            f"MAE={mae_validation_fold:.3f}, RMSE={rmse_validation_fold:.3f}, R2={r2_validation_fold:.3f}"
        )

results_nested_df = print_results_regression(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]

[fold RVAE] epoch 0 | loss 0.025559 | recon 0.025559 | kl 0.001727
[fold RVAE] epoch 300 | loss 0.011900 | recon 0.011900 | kl 0.026375
[fold RVAE] epoch 600 | loss 0.009745 | recon 0.009745 | kl 0.086084
[fold RVAE] epoch 900 | loss 0.009237 | recon 0.009237 | kl 0.127296
[fold RVAE] epoch 1200 | loss 0.008878 | recon 0.008878 | kl 0.183043
[fold RVAE] epoch 1500 | loss 0.008077 | recon 0.008077 | kl 0.279835
[fold RVAE] epoch 1800 | loss 0.007406 | recon 0.007406 | kl 0.393358
[fold RVAE] epoch 1999 | loss 0.007107 | recon 0.007106 | kl 0.517775
Fold 01 | KNN | Validation: MAE=1.520, RMSE=1.976, R2=0.837
[fold RVAE] epoch 0 | loss 0.025421 | recon 0.025421 | kl 0.002266
[fold RVAE] epoch 300 | loss 0.011335 | recon 0.011335 | kl 0.038288
[fold RVAE] epoch 600 | loss 0.009456 | recon 0.009456 | kl 0.122200
[fold RVAE] epoch 900 | loss 0.009113 | recon 0.009113 | kl 0.185060
[fold RVAE] epoch 1200 | loss 0.008921 | recon 0.008920 | kl 0.234783
[fold RVAE] epoch 1500 | loss 0.008333 | r

KeyboardInterrupt: 

In [15]:
from ci import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,1.326452,2.813791,0.740744,0.862798
ci,"[1.023, 1.617]","[2.239, 3.237]","[0.667, 0.814]","[0.819, 0.909]"


In [26]:
def train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T, R=10,
                             num_epochs=2000, lr=1e-3, verbose=False, seed=42):
    """Train a fresh KendallRVAE on a training subset only.

    X_tan_train : (N_train, D) tangent vectors
    X_man_train : (N_train, K*M*T) manifold trajectories (flattened)
    seed : random seed for model initialization
    """
    # Set seeds before model initialization for reproducibility
    deterministic(seed)
    
    D = X_tan_train.shape[1]

    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = KendallRVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    model_fold.train()
    for epoch in range(num_epochs):
        opt_fold.zero_grad(set_to_none=True)

        x_hat_man_train, mu_train, logvar_train, z_train, v_hat_train = model_fold(X_tan_train)

        dist_train = squared_geodesic_distance(X_man_train, x_hat_man_train, K, M, T)
        recon_train = torch.mean(dist_train)

        kl_train = -0.5 * torch.sum(1 + logvar_train - mu_train.pow(2) - logvar_train.exp(), dim=1)
        avg_kl_train = kl_train.mean()
        loss_train = recon_train + 1e-6 * avg_kl_train
        # loss_train = recon_train + 1e-5 * kl_train + lambda_cov * cov_penalty(mu_train)

        loss_train.backward()
        opt_fold.step()

        if verbose and (epoch % 300 == 0 or epoch == num_epochs - 1):
            print(f"[fold RVAE] epoch {epoch} | loss {loss_train:.6f} | recon {recon_train:.6f} | kl {avg_kl_train:.6f}")

    model_fold.eval()
    return model_fold

In [ ]:
# Nested CV: re-train KendallRVAE inside each participant fold

# Set random seeds for reproducibility
import random
import numpy as np
import torch

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from tqdm.notebook import tqdm


# Regression models with random_state set for reproducibility
models = {'KNN': KNeighborsRegressor()}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_poma)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 30
R = 38
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    deterministic(fold_seed)

    # Slice tangent and manifold data for this fold
    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]

    # Train fold-specific KendallRVAE on train participants only
    model_fold = train_kendall_rvae_fold(X_tan_train, X_man_train, K, M, T,
                                         R=R, num_epochs=2000, lr=1e-3, verbose=True, seed=fold_seed)

    # Extract latent means (mu) for train, validation, and test using the fold-specific encoder
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_validation_fold, _ = model_fold.vae.encode(X_tan[validation_idx])
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_validation_fold = mu_validation_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    y_train_fold = y_poma[train_idx]
    y_validation_fold = y_poma[validation_idx]
    y_test_fold = y_poma[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold validation metrics (printed)
        mae_validation_fold = mean_absolute_error(y_validation_fold, validation_preds)
        rmse_validation_fold = np.sqrt(mean_squared_error(y_validation_fold, validation_preds))
        r2_validation_fold = r2_score(y_validation_fold, validation_preds)

        print(
            f"Fold {k + 1:02d} | {name} | Validation: "
            f"MAE={mae_validation_fold:.3f}, RMSE={rmse_validation_fold:.3f}, R2={r2_validation_fold:.3f}"
        )

results_nested_df = print_results_regression(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]

[fold RVAE] epoch 0 | loss 0.025559 | recon 0.025559 | kl 0.065626
[fold RVAE] epoch 300 | loss 0.011902 | recon 0.011901 | kl 0.996583
[fold RVAE] epoch 600 | loss 0.009753 | recon 0.009750 | kl 3.245109
[fold RVAE] epoch 900 | loss 0.009242 | recon 0.009237 | kl 4.797538
[fold RVAE] epoch 1200 | loss 0.008892 | recon 0.008885 | kl 6.824241
[fold RVAE] epoch 1500 | loss 0.008078 | recon 0.008068 | kl 10.328637
[fold RVAE] epoch 1800 | loss 0.007422 | recon 0.007407 | kl 14.177569
[fold RVAE] epoch 1999 | loss 0.007277 | recon 0.007259 | kl 17.305017
Fold 01 | KNN | Validation: MAE=1.520, RMSE=1.976, R2=0.837
[fold RVAE] epoch 0 | loss 0.025421 | recon 0.025421 | kl 0.086113
[fold RVAE] epoch 300 | loss 0.011341 | recon 0.011340 | kl 1.438664
[fold RVAE] epoch 600 | loss 0.009462 | recon 0.009458 | kl 4.618517
[fold RVAE] epoch 900 | loss 0.009119 | recon 0.009112 | kl 6.943081
[fold RVAE] epoch 1200 | loss 0.008933 | recon 0.008925 | kl 8.710438
[fold RVAE] epoch 1500 | loss 0.008356 